In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np 
from sklearn.preprocessing import StandardScaler
import umap
import distinctipy
import matplotlib.pyplot as plt
import cv2
import pickle 
from pathlib import Path
from sklearn.metrics import silhouette_score
from pathlib import Path

In [ ]:
def create_color_dict(labels_unique, colors_to_exclude=None):
    color_arr = distinctipy.get_colors(len(labels_unique), colors_to_exclude)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
cell_cluster_path = r"" # path to cell phenotyping results
fibrosis_path = r"" # path fibrosis clustering results

In [ ]:
cell_fibrosis_barcode_path = # path to cell fibrosis barcode results

In [ ]:
Path.mkdir(Path(cell_fibrosis_barcode_path), exist_ok=True)

In [ ]:
keys = [] # sample identifiers

In [ ]:
keys_fibrosis_columns = {} # sample identifier : order of fibrosis subtypes

In [ ]:
master_fibrosis_column_order = ['Established Fibrosis', 'i-IFTA', 'Diffuse Fibrosis']

In [ ]:
master_marker_columns = ['CD20', 'CD8', 'CD4', 'CD68', 'PODXL', 'PANCK', 'LRP2', 'CD31']

In [ ]:
num_fibrosis_clusters = 3

In [ ]:
fibrosis_df_dict = {}
fl_fibrosis_df_dict = {}

In [ ]:
for key in tqdm(keys): 
    cell_cluster_df = pd.read_csv(cell_cluster_path + key + '/1x/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0', 'index', 'index.1'])

    with open(fibrosis_path + key + '/pixel_counts_outside_cells.pickle', 'rb') as f: 
            fibrosis_counts = pickle.load(f)
        
    fibrosis_df = pd.DataFrame(fibrosis_counts, columns=range(1, int(num_fibrosis_clusters)+1))
    fibrosis_df.columns = keys_fibrosis_columns[key]
    fibrosis_df = fibrosis_df[master_fibrosis_column_order]
    fibrosis_df = fibrosis_df.div(fibrosis_df.sum(axis=1), axis=0)
    fibrosis_df['Cell ID'] = cell_cluster_df['Cell ID']
    fibrosis_df['X'] = cell_cluster_df['Y']
    fibrosis_df['Y'] = cell_cluster_df['X']
    fibrosis_df['Cell Type'] = cell_cluster_df['Cell Type']
    fibrosis_df = fibrosis_df.dropna()

    fibrosis_high_df = fibrosis_df.loc[fibrosis_df.loc[:,master_fibrosis_column_order[:-1]].sum(axis=1)>0.9, :]

    fibrosis_df_dict[key] = fibrosis_df

    cell_by_fibrosis_df = (cell_cluster_df.loc[cell_cluster_df['Cell ID'].isin(list(fibrosis_df['Cell ID']))].reindex(columns=master_marker_columns, fill_value=0))
    cell_by_high_fibrosis_df = (cell_cluster_df.loc[cell_cluster_df['Cell ID'].isin(list(fibrosis_high_df['Cell ID']))].reindex(columns=master_marker_columns, fill_value=0))


    fl_fibrosis_df_dict[key] = pd.concat([cell_by_fibrosis_df, fibrosis_df], axis=1)


In [ ]:
key_condition_dict = {} # sample identifier : sample condition 

In [ ]:
for key, df in fibrosis_df_dict.items():
    df['Condition'] = key_condition_dict[key]
    fibrosis_df_dict[key] = df

In [ ]:
for key, df in fl_fibrosis_df_dict.items():
    df['Condition'] = key_condition_dict[key]
    fl_fibrosis_df_dict[key] = df

In [ ]:
n_master_columns = 9
n_cell_master_columns = 17

In [ ]:
n = n_master_columns-5
stacked = [
        df.iloc[:, :n]
        for df in fibrosis_df_dict.values()
    ]

fibrosis_arr = np.array(pd.concat(stacked, axis=0, ignore_index=True))


In [ ]:
stacked = [
    df.loc[
        ~df['Cell Type'].astype(str).eq('Unclassified'),
        df.columns[:n]
    ]
    for df in fibrosis_df_dict.values()
]

fibrosis_classified_arr = np.array(pd.concat(stacked, axis=0, ignore_index=True))

In [ ]:
n_cell = n_cell_master_columns-5

stacked = [
    df.loc[
        ~df['Cell Type'].astype(str).eq('Unclassified'),
        df.columns[:n_cell]
    ]
    for df in fl_fibrosis_df_dict.values()
]

fl_fibrosis_classified_arr = np.array(pd.concat(stacked, axis=0, ignore_index=True))

In [ ]:
n_cell_only = n_cell_master_columns-5-4

stacked = [
    df.loc[
        ~df['Cell Type'].astype(str).eq('Unclassified'),
        df.columns[:n_cell_only]
    ]
    for df in fl_fibrosis_df_dict.values()
]

fl_only_fibrosis_classified_arr = np.array(pd.concat(stacked, axis=0, ignore_index=True))


In [ ]:
condition_classified = [
    df.loc[
        ~df['Cell Type'].astype(str).eq('Unclassified'),
        'Condition'
    ]
    for df in fibrosis_df_dict.values()
]
condition_classified = np.array(pd.concat(condition_classified, axis=0, ignore_index=True))

In [ ]:
condition_color_dict = create_color_dict(list(set(list(key_condition_dict.values()))))
condition_classified_colors = [condition_color_dict[s] for s in condition_classified]

In [ ]:
scaler = StandardScaler()
fl_fibrosis_classified_arr_scaled = scaler.fit_transform(fl_fibrosis_classified_arr)

umap_fl_fibrosis_classified_model = umap.UMAP(n_components=2) 
umap_fl_fibrosis_classified = umap_fl_fibrosis_classified_model.fit_transform(fl_fibrosis_classified_arr_scaled)

In [ ]:
scaler = StandardScaler()
fl_only_fibrosis_classified_arr_scaled = scaler.fit_transform(fl_only_fibrosis_classified_arr)
umap_fl_only_fibrosis_classified_model = umap.UMAP(n_components=2) 
umap_fl_only_fibrosis_classified = umap_fl_fibrosis_classified_model.fit_transform(fl_only_fibrosis_classified_arr_scaled)

In [ ]:
scaler = StandardScaler()
fibrosis_classified_arr_scaled = scaler.fit_transform(fibrosis_classified_arr)
umap_fibrosis_classified_model = umap.UMAP(n_components=2) 
umap_fibrosis_classified = umap_fibrosis_classified_model.fit_transform(fibrosis_classified_arr_scaled)

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.size'] = 25
fig, ax = plt.subplots(1)
fig.set_figheight(15)
fig.set_figwidth(15)
ax.scatter(umap_fibrosis_classified[:, 0], umap_fibrosis_classified[:, 1], s=1, c=condition_classified_colors)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def neighborhood_purity(X, y, k=6000):
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
    indices = nbrs.kneighbors(X, return_distance=False)[:, 1:]
    purity = [(y[idx] == y[i]).mean() for i, idx in enumerate(indices)]
    return np.mean(purity)

In [ ]:
f1_fibrosis_classified_purity = neighborhood_purity(fl_fibrosis_classified_arr, condition_classified)
f1_only_fibrosis_classified_purity = neighborhood_purity(fl_only_fibrosis_classified_arr, condition_classified)
fibrosis_classified_purity = neighborhood_purity(fibrosis_classified_arr, condition_classified)

print(f1_fibrosis_classified_purity)
print(f1_only_fibrosis_classified_purity)
print(fibrosis_classified_purity)